In [2]:
# Mount Google Drive to access your dataset
from google.colab import drive
drive.mount('/content/drive')

# Import pandas for data handling
import pandas as pd

# Path to your cleaned dataset
file_path = '/content/drive/MyDrive/Scam Dataset/phishing_cleaned.csv'

# Load dataset into a DataFrame
df = pd.read_csv(file_path)

# Display first few rows to confirm loading
df.head()

Mounted at /content/drive


,url,ranking,isip,valid,activeduration,urllen,is@,isredirect,havedash,domainlen,nosofsubdomain,label
0,www.zvon.org/xxl/WSDL1.1/Output/index.html,194914,0,1,7305,42,0,0,0,12,2,0
1,bima.astro.umd.edu/nemo/linuxastro/,7001,0,0,0,35,0,0,0,18,3,0
2,www.synchrotech.com/support/install.html,10000000,0,1,12053,40,0,0,0,19,2,0
3,www.ansi.okstate.edu/breeds/swine/largeblackwh...,23191,0,0,0,50,0,0,0,20,3,0
4,www.strum.co.uk/webbery/,10000000,0,0,0,24,0,0,0,15,3,0


In [3]:
# Import required libraries for URL parsing and regex
import re
from urllib.parse import urlparse

# Function to extract meaningful features from a URL
def extract_features(url):
    features = {}

    # Parse the URL into components (scheme, domain, etc.)
    parsed = urlparse(str(url))

    # Basic structural features
    features['url_length'] = len(url)  # Total length of URL
    features['num_dots'] = url.count('.')  # Number of dots in URL
    features['num_hyphens'] = url.count('-')  # Number of hyphens
    features['num_at'] = url.count('@')  # Presence of '@' symbol
    features['num_digits'] = sum(c.isdigit() for c in url)  # Count digits
    features['num_special_chars'] = len(re.findall(r'[^a-zA-Z0-9]', url))  # Special characters

    # Domain-based features
    features['has_https'] = 1 if parsed.scheme == 'https' else 0  # HTTPS usage
    features['num_subdomains'] = len(parsed.netloc.split('.')) - 2 if parsed.netloc else 0  # Subdomains

    # Suspicious keywords commonly found in phishing URLs
    suspicious_words = ['login', 'verify', 'bank', 'secure', 'account', 'update']
    features['has_suspicious_words'] = int(any(word in url.lower() for word in suspicious_words))

    return features

# Apply feature extraction to all URLs
features_df = df['url'].apply(lambda x: pd.Series(extract_features(x)))

# Combine extracted features with labels
data = pd.concat([features_df, df['label']], axis=1)

# Display processed data
data.head()

,url_length,num_dots,num_hyphens,num_at,num_digits,num_special_chars,has_https,num_subdomains,has_suspicious_words,label
0,42,4,0,0,2,8,0,0,0,0
1,35,3,0,0,0,6,0,0,0,0
2,40,3,0,0,0,5,0,0,0,0
3,50,3,0,0,0,7,0,0,0,0
4,24,3,0,0,0,5,0,0,0,0


In [5]:
# Separate features (X) and target (y)
X = data.drop('label', axis=1).values  # Convert to NumPy array
y = data['label'].values

In [6]:
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Shuffle dataset indices
indices = np.arange(len(X))
np.random.shuffle(indices)

# Define split point (80% training, 20% testing)
split = int(0.8 * len(X))

# Split data manually
X_train, X_test = X[indices[:split]], X[indices[split:]]
y_train, y_test = y[indices[:split]], y[indices[split:]]

In [8]:
# Function to calculate Gini Impurity
def gini(y):
    classes = np.unique(y)
    impurity = 1
    for c in classes:
        p = np.sum(y == c) / len(y)
        impurity -= p ** 2
    return impurity

In [9]:
# Function to split dataset based on feature and threshold
def split_dataset(X, y, feature, threshold):
    left_mask = X[:, feature] < threshold
    right_mask = X[:, feature] >= threshold

    return X[left_mask], X[right_mask], y[left_mask], y[right_mask]

In [11]:
# Simple Decision Tree Class
class DecisionTree:
    def __init__(self, max_depth=3):
        self.max_depth = max_depth

    # Train the tree
    def fit(self, X, y, depth=0):
        # If all labels are the same, return that label
        if len(set(y)) == 1:
            return y[0]

        # Stop if max depth reached
        if depth >= self.max_depth:
            return np.bincount(y).argmax()

        best_feature, best_thresh, best_gain = None, None, -1

        # Try all features and thresholds
        for feature in range(X.shape[1]):
            thresholds = np.unique(X[:, feature])

            for t in thresholds:
                X_l, X_r, y_l, y_r = split_dataset(X, y, feature, t)

                if len(y_l) == 0 or len(y_r) == 0:
                    continue

                # Compute information gain
                gain = gini(y) - (
                    len(y_l)/len(y)*gini(y_l) +
                    len(y_r)/len(y)*gini(y_r)
                )

                if gain > best_gain:
                    best_feature, best_thresh, best_gain = feature, t, gain

        # If no split improves, return majority class
        if best_gain == -1:
            return np.bincount(y).argmax()

        # Create child nodes recursively
        X_l, X_r, y_l, y_r = split_dataset(X, y, best_feature, best_thresh)

        return {
            'feature': best_feature,
            'threshold': best_thresh,
            'left': self.fit(X_l, y_l, depth+1),
            'right': self.fit(X_r, y_r, depth+1)
        }

    # Predict one sample
    def predict_one(self, x, tree):
        if not isinstance(tree, dict):
            return tree

        if x[tree['feature']] < tree['threshold']:
            return self.predict_one(x, tree['left'])
        else:
            return self.predict_one(x, tree['right'])

    # Predict multiple samples
    def predict(self, X, tree):
        return np.array([self.predict_one(x, tree) for x in X])

In [12]:
# Bagging Classifier
class BaggingClassifier:
    def __init__(self, n_trees=10, max_depth=3):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.trees = []

    # Train multiple trees using bootstrap sampling
    def fit(self, X, y):
        self.trees = []

        for _ in range(self.n_trees):
            # Bootstrap sampling (sampling with replacement)
            indices = np.random.choice(len(X), len(X), replace=True)
            X_sample = X[indices]
            y_sample = y[indices]

            # Train a tree
            tree = DecisionTree(max_depth=self.max_depth)
            trained_tree = tree.fit(X_sample, y_sample)

            self.trees.append(trained_tree)

    # Predict using majority voting
    def predict(self, X):
        predictions = []

        for tree in self.trees:
            dt = DecisionTree()
            predictions.append(dt.predict(X, tree))

        # Convert to array and take majority vote
        predictions = np.array(predictions)
        final_pred = np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=predictions)

        return final_pred

In [14]:
# Initialize Bagging model
bagging_model = BaggingClassifier(n_trees=15, max_depth=3)

# Train model
bagging_model.fit(X_train, y_train)

In [15]:
# Predict on test data
y_pred = bagging_model.predict(X_test)

# Calculate accuracy
accuracy = np.mean(y_pred == y_test)

print("Accuracy:", accuracy)

Accuracy: 0.8049153908138598


In [17]:
# Install widgets if not already available
!pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.1 MB/s eta 0:00:00


In [19]:
import ipywidgets as widgets
from IPython.display import display, clear_output

In [20]:
# Text box for user input
url_input = widgets.Text(
    value='',
    placeholder='Paste a URL here...',
    description='URL:',
    layout=widgets.Layout(width='80%')
)

# Predict button
predict_button = widgets.Button(
    description="Predict",
    button_style='success'
)

# Clear button
clear_button = widgets.Button(
    description="Clear",
    button_style='warning'
)

# Output area
output_area = widgets.Output()

In [21]:
# Function for prediction
def on_predict_clicked(b):
    with output_area:
        clear_output()

        url = url_input.value.strip()

        # Handle empty input
        if url == "":
            print("⚠️ Please enter a URL.")
            return

        # Extract features and predict
        feat = extract_features(url)
        feat_array = np.array(list(feat.values())).reshape(1, -1)

        pred = bagging_model.predict(feat_array)[0]

        if pred == 1:
            print("⚠️ Phishing URL detected")
        else:
            print("✅ Legitimate URL")

In [26]:
# Function to clear input and output
def on_clear_clicked(b):
    url_input.value = ""
    with output_area:
        clear_output()

In [23]:
predict_button.on_click(on_predict_clicked)
clear_button.on_click(on_clear_clicked)

In [34]:
display(url_input, widgets.HBox([predict_button, clear_button]), output_area)

⚠️ Phishing URL detected
